The initial plan was to train our own version of GloVe and play with it a little. Unfortunately, Fedor couldn't figure out how to do it (he knows how to do it in R, but apparently it's not trivial in Python). Sorry about that!

Note that the whole point of training GloVe is that it is an unsupervised model. It means that we do not need annotated data to train it, i.e., we can simply download the whole Wikipedia or a collection of books from a library and just train. Later, when we need to use word embeddings for, say, classification, we can combine GloVe with a supervised model and train it on a much smaller annotated dataset.

For your project, if you need word embeddings, you can use GloVe, but you need to make sure that you are using the correct version of word embeddings. For example, if you are going to analyse tweets in your project, then load the word embeddings trained on twitter.

You are also encouraged to explore other word embedding models, such as word2vec. It is actually easier to train since the necessary functions are available in the library `gensim`.

> Блок с отступами



In [3]:
# This is to download pre-trained word vectors:

!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

# You can comment it out later

--2025-01-25 06:14:02--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-01-25 06:14:02--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-01-25 06:14:02--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

# Notebook 6 - playing with GloVe further

We will explore GloVe further here. There will be some math questions in the end - please try them.

In [4]:
#@title
# Libraries:
import pandas as pd
import numpy as np

Here we download pretrained GloVe and explore word vectors. Some useful functions will be provided for you.

In [5]:
# Word vectors:

path_to_glove_file = "glove.6B.300d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print("Found %s word vectors." % len(embeddings_index))

Found 400000 word vectors.


We have created a Python dictionary with words as keys and word vectors as values.

In [6]:
print("Type of the variable with word embeddings is", type(embeddings_index), "\n")
print("Here are the first 10 keys")
print(list(embeddings_index.keys())[:10], "\n")
print("First 5 entries of the vector v('engineer'):")
print(embeddings_index.get('engineer', 0)[:5])

Type of the variable with word embeddings is <class 'dict'> 

Here are the first 10 keys
['the', ',', '.', 'of', 'to', 'and', 'in', 'a', '"', "'s"] 

First 5 entries of the vector v('engineer'):
[ 0.30628 -0.14603 -0.22847 -0.46993 -0.96826]


Note that we downloaded word embeddings as a Python dictionary. However, in many occasions, it may be better to work with it in a matrix form. We will now convert it to a `pandas` dataframe that is essentially the same as a matrix (Fedor tried converting this dictionary to a `numpy` matrix but this seems to be a much slower operation).

Note the option `orient = 'index'`. It produces the matix whose rows represent word vectors and rows are indexed by words themselves. If we used `orient = 'columns'`, then we would get the transposed matrix, whose columns represent word vectors and names of the columns would be words themselves.

In [7]:
df_glove = pd.DataFrame.from_dict(embeddings_index, orient='index', dtype=None)
df_glove

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
the,0.046560,0.213180,-0.007436,-0.458540,-0.035639,0.236430,-0.288360,0.215210,-0.134860,-1.641300,...,-0.013064,-0.296860,-0.079913,0.195000,0.031549,0.285060,-0.087461,0.009061,-0.209890,0.053913
",",-0.255390,-0.257230,0.131690,-0.042688,0.218170,-0.022702,-0.178540,0.107560,0.058936,-1.385400,...,0.075968,-0.014359,-0.073794,0.221760,0.146520,0.566860,0.053307,-0.232900,-0.122260,0.354990
.,-0.125590,0.013630,0.103060,-0.101230,0.098128,0.136270,-0.107210,0.236970,0.328700,-1.678500,...,0.060148,-0.156190,-0.119490,0.234450,0.081367,0.246180,-0.152420,-0.342240,-0.022394,0.136840
of,-0.076947,-0.021211,0.212710,-0.722320,-0.139880,-0.122340,-0.175210,0.121370,-0.070866,-1.572100,...,-0.366730,-0.386030,0.302900,0.015747,0.340360,0.478410,0.068617,0.183510,-0.291830,-0.046533
to,-0.257560,-0.057132,-0.671900,-0.380820,-0.364210,-0.082155,-0.010955,-0.082047,0.460560,-1.847700,...,-0.012806,-0.597070,0.317340,-0.252670,0.543840,0.063007,-0.049795,-0.160430,0.046744,-0.070621
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chanty,0.392700,-0.022505,0.304580,0.187990,0.141180,0.724030,-0.257810,-0.137290,-0.016521,0.595960,...,-0.182950,0.406630,-0.343630,-0.270400,-0.593680,0.016447,0.140740,0.463940,-0.369570,-0.287180
kronik,0.136790,-0.139090,-0.360890,0.079864,0.321490,0.263870,-0.109900,0.044420,0.083869,0.791330,...,0.036419,-0.036845,-0.348150,0.064732,-0.000577,-0.133790,0.428960,-0.023320,0.410210,-0.393080
rolonda,0.075713,-0.040502,0.183450,0.512300,-0.228560,0.839110,0.178780,-0.713010,0.326900,0.695350,...,-0.388530,0.545850,-0.035050,-0.184360,-0.197000,-0.350030,0.160650,0.218380,0.309670,0.437610
zsombor,0.814510,-0.362210,0.311860,0.813810,0.188520,-0.313600,0.827840,0.296560,-0.085519,0.475970,...,0.130880,0.106120,-0.408110,0.313380,-0.430250,0.069798,-0.207690,0.075486,0.284080,-0.175590


Here is how we can extract a word vector from this matrix:

In [8]:
print("Vector for 'China is (first 5 entries):'")
df_glove.loc['china', :][:5]
print("||v(China)|| =", np.linalg.norm(df_glove.loc['china', :]))

Vector for 'China is (first 5 entries):'
||v(China)|| = 7.532856


The cosine similarity between two vectors $x$ and $y$ is
$$
\cos\alpha = \frac{x\cdot y}{\|x\|\cdot \|y\|}
$$

In [9]:
def cosine_sim(x, y):
    return np.dot(x, y) / np.sqrt(np.dot(x, x) * np.dot(y, y))

print("Cosine similarity between 'china' and 'singapore' is ", \
      cosine_sim(df_glove.loc['china', :], df_glove.loc['singapore', :]))


Cosine similarity between 'china' and 'singapore' is  0.45664227


Let's say that now we want to compute the cosine similarity of any given vector, say, $v(\mbox{China})$, with all word vectors stored in our matrix in one go. This is essentially a matrix multiplication and here is how we can do it.

We begin with computing the dot products of all vectors with the given vector

In [10]:
x = df_glove.loc['china', :]
dot_products = np.dot(df_glove, x)
print("Sample of the dot products")
print(dot_products[:5])

Sample of the dot products
[13.532056  13.61245   12.294808  14.667965  15.9136715]


Note that this is an unnamed vector (Fedor doesn't know why it lost the names). Let's now find norms of all the word vectors. Here is how we do it:

In [11]:
word_vector_norms = np.sqrt(np.square(df_glove).sum(1))
print("Sample of word vector norms:")
print(word_vector_norms[:5])
print("Sanity check: ||v(China|| =", word_vector_norms.loc["china"])

Sample of word vector norms:
the    5.181397
,      4.347232
.      4.458036
of     5.454492
to     5.914590
dtype: float32
Sanity check: ||v(China|| = 7.5328565


Notice that the vector of word vector norms is named and we can extract its elements by the name. This is nice!

Now we can find all the cosine similarities in one go!

In [12]:
all_cosine_similarities = dot_products / (word_vector_norms * np.linalg.norm(df_glove.loc['china', :]))
all_cosine_similarities

,0
the,0.346703
",",0.415684
.,0.366116
of,0.356990
to,0.357179
...,...
chanty,-0.239734
kronik,-0.269540
rolonda,-0.268389
zsombor,-0.121170


And now if we want to identify words most similar to the given one, it remains to sort this and print the top most similar words

In [13]:
print("Top 10 words most similar to 'China':")
all_cosine_similarities[np.argsort(-all_cosine_similarities)][:10]

Top 10 words most similar to 'China':


<ipython-input-13-4e965f65b247>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  all_cosine_similarities[np.argsort(-all_cosine_similarities)][:10]


,0
china,1.000000
chinese,0.788698
beijing,0.772798
taiwan,0.681081
shanghai,0.624310
mainland,0.623092
guangdong,0.609352
tibet,0.581537
hong,0.581442
kong,0.575354


Now we will put it together into a single function

In [14]:
def top_most_similar_words(x, n = 10, glove_matrix = df_glove):
    # Returns the table of top n words with highest cosine similarity to vector x
    dot_products = np.dot(glove_matrix, x)
    word_vector_norms = np.sqrt(np.square(glove_matrix).sum(1))
    x_norm = np.linalg.norm(x)
    all_cosine_similarities = dot_products / (word_vector_norms * x_norm)
    return all_cosine_similarities[np.argsort(-all_cosine_similarities)][:n]

print("Sanity check: top 5 words most similar to 'China':")
top_most_similar_words(df_glove.loc['china', :], 5)

Sanity check: top 5 words most similar to 'China':


<ipython-input-14-494341afa719>:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return all_cosine_similarities[np.argsort(-all_cosine_similarities)][:n]


,0
china,1.000000
chinese,0.788698
beijing,0.772798
taiwan,0.681081
shanghai,0.624310


### Exercise 1

It is important to understand the corpus that our vector embeddings were trained on. For example, the word "apple" can be either a fruit or the tech company / computer. Which of the two meanings is more typical to the corpus that our pretrained word embeddings were trained on?

In [15]:
top_most_similar_words(df_glove.loc['apple', :], 20)

<ipython-input-14-494341afa719>:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return all_cosine_similarities[np.argsort(-all_cosine_similarities)][:n]


,0
apple,1.000000
iphone,0.598704
macintosh,0.583633
ipod,0.576112
microsoft,0.566383
ipad,0.562810
intel,0.545756
ibm,0.528619
google,0.528247
imac,0.507252


Judging by this, the corpus usually mentions "apple" in the context of tech companies and computers.

__END OF EXERCISE 1__

GloVe is designed to capture semantic relationships between words. For example, we expect the following relation:
$$
v(\mbox{China})+v(\mbox{capital})=v(\mbox{Beijing})
$$
And it works:

In [16]:
top_most_similar_words(df_glove.loc['china', :] + df_glove.loc['capital', :], 5)

<ipython-input-14-494341afa719>:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return all_cosine_similarities[np.argsort(-all_cosine_similarities)][:n]


,0
china,0.814104
capital,0.800777
beijing,0.717299
chinese,0.701161
shanghai,0.591910


### Exercise 2

We also expect more complicated relationships, such as
$$
v(\mbox{father})-v(\mbox{mother})=v(\mbox{man})-v(\mbox{woman})
$$
Identify words that are most similar to
$$
v(\mbox{father})-v(\mbox{man})+v(\mbox{woman})
$$
Will you get "mother"?

In [17]:
top_most_similar_words(df_glove.loc['father', :] + df_glove.loc['woman', :]-df_glove.loc['man', :], 5)

<ipython-input-14-494341afa719>:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return all_cosine_similarities[np.argsort(-all_cosine_similarities)][:n]


,0
mother,0.835898
daughter,0.793933
father,0.764212
husband,0.733081
wife,0.732290


### Exercise 3

Now let's get closer to finance. What should be the word "?" in the following relation?
$$
v(\mbox{york})-?=v(\mbox{nasdaq})-v(\mbox{nikkei})
$$
Will GloVe be able to figure it out?

In [18]:
top_most_similar_words(df_glove.loc['york', :] + df_glove.loc['nikkei', :]-df_glove.loc['nasdaq', :], 5)

<ipython-input-14-494341afa719>:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return all_cosine_similarities[np.argsort(-all_cosine_similarities)][:n]


,0
nikkei,0.597821
york,0.555420
tokyo,0.450557
new,0.399001
shimbun,0.391373


### Exercise 4

When working with word embeddings, it may be important to pay attention to biases in the corpus.

For example, it is found [here](https://proceedings.neurips.cc/paper/2016/file/a486cd07e4ac3d270571622f4f316ec5-Paper.pdf) that
$$
v(\mbox{man})-v(\mbox{woman})\approx v(\mbox{computer programmer})-v(\mbox{homemaker})
$$
Sometimes, such biases may pose a serious issue if the machine learning model for solving the end taks still retains such a bias. Imagine that you are designing a model that recommends a career path for a financial firm trainee and recommended career paths are different for men and women.

Let's check our corpus for biases. We will work with the following list of jobs (this is some random list Fedor composed):

In [19]:
job_words = ['accountant', 'manager', 'secretary', 'programmer', \
             'engineer', 'teller', 'analyst', 'receptionist', 'intern',\
             'officer', 'actuary', 'trader', 'sales', 'director']

print(job_words)

['accountant', 'manager', 'secretary', 'programmer', 'engineer', 'teller', 'analyst', 'receptionist', 'intern', 'officer', 'actuary', 'trader', 'sales', 'director']


In [26]:
def top_words(word, words, glove_matrix=df_glove):
    # Compute dot products between the target word and all words in GloVe matrix
    dot_products = np.dot(glove_matrix, glove_matrix.loc[word, :])

    # Compute norms of all word vectors
    word_vector_norms = np.sqrt(np.square(glove_matrix).sum(axis=1))

    # Compute norm of the target word vector
    word_norm = np.linalg.norm(glove_matrix.loc[word, :])

    # Compute cosine similarities
    all_cosine_similarities = dot_products / (word_vector_norms * word_norm)

    # Extract relevant words and return as a DataFrame
    df_similarity = pd.DataFrame({
        'word': words,
        'cosine_similarity': all_cosine_similarities.loc[words]
    }).set_index('word').sort_values(by='cosine_similarity', ascending=False)

    return df_similarity


Print words most associated with "he" and words most associated with "she". Is there evidence for bias here?

In [27]:
top_words('he',job_words,df_glove)

,cosine_similarity
word,
officer,0.378355
manager,0.369735
director,0.303555
secretary,0.290478
engineer,0.287143
sales,0.196833
accountant,0.140302
trader,0.125802
analyst,0.105503


In [28]:
top_words('she',job_words,df_glove)

,cosine_similarity
word,
officer,0.312915
director,0.283813
manager,0.234443
secretary,0.232065
receptionist,0.213460
sales,0.174145
intern,0.165595
engineer,0.143594
accountant,0.142416


There may be some gender bias here since "receptionist" does not appear in the male list in the top 10 but in the female list it is number 5. But it is not strong.

# Theoretical homework

### Question 1

The GloVe loss function is
$$
L(w,\tilde{w}, b, \tilde{b})=\sum_{i,k=1}^{V}f(X_{ik})\left(w_i^{T} \tilde{w}_k + b_i + \tilde{b}_k - \log X_{ik} \right)^2,
$$
Derive its gradient, i.e., find
$$
\frac{\partial L}{\partial w_i},
\frac{\partial L}{\partial \tilde{w}_k},
\frac{\partial L}{\partial b_i},
\frac{\partial L}{\partial \tilde{b}_k}
$$

***ANS***<br>
We will compute the gradients of the **GloVe loss function** step by step.

### **Given Loss Function:**
L(w,\tilde{w}, b, \tilde{b}) = \sum_{i,k=1}^{V} f(X_{ik})\left(w_i^{T} \tilde{w}_k + b_i + \tilde{b}_k - \log X_{ik} \right)^2
where:
- \( w_i \) and \( \tilde{w}_k \) are word vectors.
- \( b_i \) and \( \tilde{b}_k \) are bias terms.
- \( f(X_{ik}) \) is a weighting function.
- \( X_{ik} \) is the co-occurrence count of words \( i \) and \( k \).

---

## **Step 1: Define the Residual Error**
Define:
\[
E_{ik} = w_i^T \tilde{w}_k + b_i + \tilde{b}_k - \log X_{ik}
\]
so that the loss function simplifies to:
\[
L = \sum_{i,k} f(X_{ik}) E_{ik}^2.
\]

---

## **Step 2: Compute Gradients**
We differentiate \( L \) with respect to each parameter.

### **1. Gradient w.r.t. \( w_i \)**
\[
\frac{\partial L}{\partial w_i} = \sum_k f(X_{ik}) \cdot 2 E_{ik} \cdot \frac{\partial E_{ik}}{\partial w_i}
\]
Since \( E_{ik} = w_i^T \tilde{w}_k + b_i + \tilde{b}_k - \log X_{ik} \), we compute:
\[
\frac{\partial E_{ik}}{\partial w_i} = \tilde{w}_k.
\]
Thus,
\[
\frac{\partial L}{\partial w_i} = \sum_k 2 f(X_{ik}) E_{ik} \tilde{w}_k.
\]

---

### **2. Gradient w.r.t. \( \tilde{w}_k \)**
\[
\frac{\partial L}{\partial \tilde{w}_k} = \sum_i 2 f(X_{ik}) E_{ik} w_i.
\]

---

### **3. Gradient w.r.t. \( b_i \)**
\[
\frac{\partial L}{\partial b_i} = \sum_k 2 f(X_{ik}) E_{ik} \cdot \frac{\partial E_{ik}}{\partial b_i}.
\]
Since \( \frac{\partial E_{ik}}{\partial b_i} = 1 \), we get:
\[
\frac{\partial L}{\partial b_i} = \sum_k 2 f(X_{ik}) E_{ik}.
\]

---

### **4. Gradient w.r.t. \( \tilde{b}_k \)**
\[
\frac{\partial L}{\partial \tilde{b}_k} = \sum_i 2 f(X_{ik}) E_{ik}.
\]

---

## **Final Gradients:**
\[
\frac{\partial L}{\partial w_i} = \sum_k 2 f(X_{ik}) E_{ik} \tilde{w}_k.
\]
\[
\frac{\partial L}{\partial \tilde{w}_k} = \sum_i 2 f(X_{ik}) E_{ik} w_i.
\]
\[
\frac{\partial L}{\partial b_i} = \sum_k 2 f(X_{ik}) E_{ik}.
\]
\[
\frac{\partial L}{\partial \tilde{b}_k} = \sum_i 2 f(X_{ik}) E_{ik}.
\]

These gradients are used to update the word vectors and biases during training.

Let me know if you need further clarifications! 🚀


Alternative weight function: f(x) = arctan(x)/arctan(x_max)

### Question 2

The weight function $f(x)$ in the GloVe loss is designed is designed so that words that
seldom co-occur do not affect it too much. It is convenient to assume that $f$ is a continuous increasing function
such that $f(0)=0$. Further, we do not want to put too much weight on words that co-occur very frequently and hence we require that
$$
\lim_{x\to\infty}f(x)=1
$$

* Show that the weight function
$$
f(x)=\begin{cases}
\left(\frac{x}{x_{max}}\right)^{\alpha},& x\le x_{max} \\
1, & x>x_{max}
\end{cases}
$$
from the original GloVe paper satisfies these conditions.
Here, $\alpha >0$ and $x_{max}$ are hyper-parameters of GloVe.

* Which trigonometric function can we use as an alternative weight function for GloVe?

***ANS:***<BR>
We will verify that the given weight function:

\[
f(x) =
\begin{cases}
\left(\frac{x}{x_{\max}}\right)^{\alpha}, & x \leq x_{\max} \\
1, & x > x_{\max}
\end{cases}
\]

satisfies the desired conditions:

1. \( f(x) \) is continuous and increasing.
2. \( f(0) = 0 \).
3. \( \lim_{x\to\infty} f(x) = 1 \).

---

### **Step 1: Checking Continuity**
To ensure \( f(x) \) is continuous at \( x = x_{\max} \), we check:

\[
\lim_{x \to x_{\max}^{-}} f(x) = \left(\frac{x_{\max}}{x_{\max}}\right)^{\alpha} = 1.
\]

\[
\lim_{x \to x_{\max}^{+}} f(x) = 1.
\]

Since both limits match, \( f(x) \) is continuous at \( x = x_{\max} \). Additionally, for all \( x < x_{\max} \), the function follows a power law, which is continuous by definition.

---

### **Step 2: Checking \( f(0) = 0 \)**
For \( x = 0 \):

\[
f(0) = \left(\frac{0}{x_{\max}}\right)^{\alpha} = 0.
\]

Thus, the condition \( f(0) = 0 \) is satisfied.

---

### **Step 3: Checking Monotonicity (Increasing Property)**
For \( 0 \leq x \leq x_{\max} \), we check the derivative:

\[
\frac{d}{dx} f(x) = \alpha \left(\frac{x}{x_{\max}}\right)^{\alpha - 1} \cdot \frac{1}{x_{\max}}.
\]

Since \( \alpha > 0 \) and \( x_{\max} > 0 \), the derivative is positive, meaning \( f(x) \) is strictly increasing for \( x \leq x_{\max} \). For \( x > x_{\max} \), \( f(x) = 1 \), which is constant and does not decrease.

Thus, \( f(x) \) is increasing.

---

### **Step 4: Checking \( \lim_{x \to \infty} f(x) = 1 \)**
For \( x > x_{\max} \), the function is explicitly defined as \( f(x) = 1 \), so:

\[
\lim_{x\to\infty} f(x) = 1.
\]

Thus, the condition is satisfied.

---

### **Conclusion**
The function satisfies all the required conditions:
1. \( f(x) \) is **continuous**.
2. \( f(x) \) is **increasing** for \( x \geq 0 \).
3. \( f(0) = 0 \).
4. \( \lim_{x\to\infty} f(x) = 1 \).

Thus, this weight function meets all the design requirements from the GloVe paper. ✅ 🚀

### Question 3

Consider the following corpus (documents are sentences):

<tt>
I do not mind wearing face masks.
I hate face shields.
I face huge challenges.
</tt>

a) Compute the term co-occurrence matrix
with the window size $2$ and weights $(1,1/2)$, i.e.,
count terms that appear next to each other with weight
$1$ and terms that appear within a distance 2 with
weight $1/2$.

b) How many nonzero summands does the GloVe loss function have?

c) Given that $d=1$ (we are constructing $1$-dimensional
word embeddings), that all biases are zero, and that the
weight function is
$$
f(x)=\begin{cases}
x , & x<1 \\
1, & x\ge 1
\end{cases}
$$
find
$$
\frac{\partial L}{\partial w_{\mbox{I}} }
$$